In [22]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

In [23]:
df = pd.read_parquet('../data/processed/text_processed.parquet')

In [24]:
df.dtypes

Ticket ID                                int64
Customer Name                           object
Customer Email                          object
Customer Age                             int64
Customer Gender                         object
Product Purchased                       object
Date of Purchase                datetime64[ns]
Ticket Type                             object
Ticket Subject                          object
Ticket Description                      object
Ticket Status                           object
Resolution                              object
Ticket Priority                         object
Ticket Channel                          object
First Response Time             datetime64[ns]
Time to Resolution              datetime64[ns]
Customer Satisfaction Rating           float64
clean_description                       object
clean_ticket_subject                    object
processed_description                   object
processed_ticket_subject                object
combined_text

In [25]:
df.head()

,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,...,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating,clean_description,clean_ticket_subject,processed_description,processed_ticket_subject,combined_text
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,...,Critical,Social media,2023-06-01 12:15:36,NaT,NaN,i am having an issue with the please assist yo...,product setup,issue please assist billing zip code 71701 app...,product setup,product setup issue please assist billing zip ...
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,...,Critical,Chat,2023-06-01 16:45:38,NaT,NaN,i am having an issue with the please assist if...,peripheral compatibility,issue please assist need change existing produ...,peripheral compatibility,peripheral compatibility issue please assist n...
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,...,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0,i am facing a problem with my the is not turni...,network problem,facing problem turning working fine yesterday ...,network problem,network problem facing problem turning working...
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,...,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0,i am having an issue with the please assist if...,account access,issue please assist problem interested would l...,account access,account access issue please assist problem int...
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,...,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0,i am having an issue with the please assist no...,data loss,issue please assist note seller responsible da...,data loss,data loss issue please assist note seller resp...


In [26]:
# create the hour features for first response time and time to resolution

df['first_response_hour'] = pd.to_datetime(
    df['First Response Time']
).dt.hour

df['resolution_hour'] = pd.to_datetime(
    df['Time to Resolution']
).dt.hour

In [27]:
# ticket description features
df['char_count'] = df['Ticket Description'].str.len()

df['word_count'] = (
    df['Ticket Description']
    .str.split()
    .str.len()
)

df['avg_word_length'] = (
    df['char_count'] /
    df['word_count']
)

In [28]:
# ticket age
reference_date = df['Date of Purchase'].max()

df['ticket_age_days'] = (
    reference_date -
    df['Date of Purchase']
).dt.days

In [29]:
# encode categorical fetures

le_gender = LabelEncoder()
le_channel = LabelEncoder()
le_product = LabelEncoder()

df['Customer Gender Encoded'] = (
    le_gender.fit_transform(
        df['Customer Gender']
    )
)

df['Ticket Channel Encoded'] = (
    le_channel.fit_transform(
        df['Ticket Channel']
    )
)

df['Product Purchased Encoded'] = (
    le_product.fit_transform(
        df['Product Purchased']
    )
)

In [30]:
# flags for customer satisfaction availaibility, resoltion time
df['has_satisfaction_rating'] = (
    df['Customer Satisfaction Rating']
    .notna()
    .astype(int)
)

df['is_resolved'] = (
    (df['Ticket Status'] == 'Closed')
    .astype(int)
)

## Train/Val/Test split

In [31]:
# for ticket type

In [32]:
y_type = df['Ticket Type']

feature_cols = [
    'processed_ticket_subject',
    'processed_description'
]

X = df[feature_cols]

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y_type,
    test_size=0.3,
    random_state=42,
    stratify=y_type
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp
)

# save
X_train.to_parquet(
    '../data/processed/ticket_type/X_train.parquet',
    index=False
)

X_val.to_parquet(
    '../data/processed/ticket_type/X_val.parquet',
    index=False
)

X_test.to_parquet(
    '../data/processed/ticket_type/X_test.parquet',
    index=False
)

y_train.to_frame().to_parquet(
    '../data/processed/ticket_type/y_train.parquet',
    index=False
)

y_val.to_frame().to_parquet(
    '../data/processed/ticket_type/y_val.parquet',
    index=False
)

y_test.to_frame().to_parquet(
    '../data/processed/ticket_type/y_test.parquet',
    index=False
)

print(X_train.shape, X_val.shape, X_test.shape)


(5928, 2) (1270, 2) (1271, 2)


In [33]:
# for ticket priority prediction
y_prior = df['Ticket Priority']

feature_cols = [
    'Customer Age',
    'Customer Gender Encoded',
    'Product Purchased Encoded',
    'Ticket Channel Encoded',
    'ticket_age_days',
    'char_count',
    'word_count',
    'avg_word_length'
]
X = df[feature_cols]

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y_prior,
    test_size=0.30,
    random_state=42,
    stratify=y_prior
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

# save
X_train.to_parquet(
    '../data/processed/ticket_priority/X_train.parquet',
    index=False
)

X_val.to_parquet(
    '../data/processed/ticket_priority/X_val.parquet',
    index=False
)

X_test.to_parquet(
    '../data/processed/ticket_priority/X_test.parquet',
    index=False
)

y_train.to_frame().to_parquet(
    '../data/processed/ticket_priority/y_train.parquet',
    index=False
)

y_val.to_frame().to_parquet(
    '../data/processed/ticket_priority/y_val.parquet',
    index=False
)

y_test.to_frame().to_parquet(
    '../data/processed/ticket_priority/y_test.parquet',
    index=False
)
print(X_train.shape, X_val.shape, X_test.shape)

(5928, 8) (1270, 8) (1271, 8)


In [35]:
# for regression task
df_reg = df[
    df['resolution_hour'].notna()
].copy()

feature_cols = [
    'Customer Age',
    'Customer Gender Encoded',
    'Product Purchased Encoded',
    'Ticket Channel Encoded',
    'ticket_age_days',
    'char_count',
    'word_count',
    'avg_word_length'
]

X = df_reg[feature_cols]

y = df_reg['resolution_hour']

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42
)

# save
X_train.to_parquet(
    '../data/processed/res_time/X_train.parquet',
    index=False
)

X_val.to_parquet(
    '../data/processed/res_time/X_val.parquet',
    index=False
)

X_test.to_parquet(
    '../data/processed/res_time/X_test.parquet',
    index=False
)

y_train.to_frame().to_parquet(
    '../data/processed/res_time/y_train.parquet',
    index=False
)

y_val.to_frame().to_parquet(
    '../data/processed/res_time/y_val.parquet',
    index=False
)

y_test.to_frame().to_parquet(
    '../data/processed/res_time/y_test.parquet',
    index=False
)

print(X_train.shape, X_val.shape, X_test.shape)

(1938, 8) (415, 8) (416, 8)


In [36]:
# save full dataset
df.to_parquet(
    '../data/processed/feature_engineered.parquet',
    index=False
)

NOTE: The timestamp fields provided for First Response Time and Time to Resolution contain inconsistent chronological ordering, producing invalid negative resolution durations. Therefore, a reliable target variable for resolution time estimation could not be derived from the dataset.

## Conclusion

Feature engineering was performed to prepare the dataset for the three machine learning tasks:

1. **Ticket Type Classification (NLP)**
2. **Ticket Priority Prediction (Tabular Classification)**
3. **Resolution Time Estimation (Regression)**

Key engineered features include:

- Processed ticket subject and description
- Combined text feature for NLP models
- Character count, word count, and average word length
- Ticket age in days
- Encoded categorical variables

Additional features such as `first_response_hour`, `resolution_hour`, `has_satisfaction_rating`, and `is_resolved` were created for exploratory purposes but excluded from the final modeling datasets to prevent target leakage.

Finally, task-specific train, validation, and test datasets were created and saved for subsequent model development and evaluation.